# Imports

In [ ]:
from cytools import Polytope, Cone

In [ ]:
import sys; sys.path.append('..')
from src import cydata, Zp, diagnostics, lattice, volume

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
import numpy as np

In [ ]:
import flint

## Hard-coded Manwe's CY

In [ ]:
if False:
    # Manwe
    verts   = [[0, 0, 0, 0], [1, -1, -1, -1], [-1, 2, 1, 1], [-1, -1, 0, 0], [-1, -1, 2, 0], [-1, -1, 2, 1], [-1, 0, 0, 2], [-1, -1, 0, 2], [-1, 0, 0, 1], [-1, 0, 1, 0], [-1, -1, 0, 1], [-1, -1, 1, 0], [-1, -1, 1, 1], [-1, 0, 1, 1], [-1, 1, 1, 1], [0, -1, 0, 0]]
    heights = [0, 35, 29, 35, 31, 35, 35, 35, 15, 17, 31, 9, 21]
elif False:
    # h11=10
    verts   = [[1, 0, 0, 0], [0, 1, 0, 0], [-14, -9, -3, -1], [-3, -2, -1, 1], [0, 0, 0, 1], [0, 0, 1, 0], [-8, -5, -2, 0], [-4, -3, -1, 1], [-1, -1, 0, 1]]
    heights = [0.0, 0.0, 20.49999999999999, 0.0, -2.4999999999999964, 4.249999999999997, 6.749999999999995, 0.0, -7.999999999999995, -3.2499999999999982, -4.499999999999998, 7.249999999999999, -2.749999999999999, -5.249999999999997, 0.0]
elif True:
    # h11=16
    verts   = [[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [-32, -21, -9, -1], [-10, -7, -3, 1], [-3, -2, -1, 1], [-1, -1, 0, 1]]
    heights = [0.0, 0.0, 3.500000000000006, 0.0, 60.750000000000014, 51.750000000000014, -17.749999999999996, 0.0, -18.25, 16.500000000000014, -20.25, -20.750000000000004, 10.000000000000009, 4.5000000000000036, 24.250000000000007, 0.0, -0.24999999999999506, -3.5000000000000036, 36.0, -5.000000000000006, -3.500000000000005]
else:
    # h11=20
    verts = [[1, 0, 0, 0], [0, 1, 0, 0], [-3, -2, -2, 2], [-24, -16, -6, -1], [-12, -8, -6, 3], [-9, -6, -5, 3], [-5, -3, -3, 2], [0, 0, 0, 1], [0, 0, 1, 0]]


In [ ]:
p       = Polytope(verts)
#t       = p.triangulate(heights=heights)
#cy      = t.cy()

## Set the conifold-related info

In [ ]:
# get the conifold charge
# -----------------------
conis = list(kklt_conifolds.kklt_conifolds(p.dual(), as_class=True))
assert len(conis) == 1
q = conis[0].conifold_charge()

In [ ]:
t  = conis[0].dual_triangulation()
cy = t.cy()

In [ ]:
data = cydata.CYData.from_cy(cy, coni_curve=q)

In [ ]:
Qmax = data.h11+data.h21+4

# Study fancier methods

In [ ]:
min_N_pts = 10000

grading = np.sum(data.H_cob, axis=0)
if data.h11<20:
    p  = Zp.mindeg_pvec_gurobi(data)
    mindeg = np.dot(p,grading)

    ps = Zp.pvecs(data, min_deg=mindeg, deg_window=max(5,mindeg//100), min_N_pts=min_N_pts)
else:
    totskc = np.rint(Cone(hyperplanes=data.H_cob).tip_of_stretched_cone()).astype(int)
    print(np.dot(totskc, grading))

    ps = Zp.pvecs(data, max_deg = 237_000)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
dry = True
verbosity = 0

In [ ]:
import time

In [ ]:
good_pfvs = []
pred = []
obs  = []

t0 = time.time()
for p in tqdm(ps[:100]):
    mat, Z, Binter = Zp.coniMellipsoid(p, data)
    proj = np.hstack([ np.zeros((data.h11-1,1), dtype=int), np.eye(data.h11-1, dtype=int) ])

    A = proj@Z@Binter
    A_fl = flint.fmpz_mat(A.tolist())
    
    for gcd in tqdm(range(1,200+1)):
        # get a basis of the lattice Kperp % d = 0
        # ----------------------------------------
        B = Zp.Kperp_gcd_lattice(data, Z, Binter, gcd)

        # get the updated L-matrix
        L = np.linalg.cholesky(B.T@mat@B)
        
        # write/solve the ellipsoid problem
        # ---------------------------------
        pred.append(volume.estimate_num_cs(
            Q=gcd*Qmax,
            L=L,
            b=(Binter@B)[0],
            offset=13,
        ))
        if (pred[-1] == 0):
            pred = pred[:-1]
            continue
    
        vs = lattice.fp_iterative_lincut(
            L=L,
            Q=gcd*Qmax,
            linvec = (Binter@B)[0],
            linmin = 13,
            max_N_out=10_000_000,
            eps=1e-4)[0]
        obs.append(len(vs))
        if len(vs) == 10_000_000:
            print('satd')
        #print(len(vs),end=',')
    
        # convert all good lattice vectors to c coefficients
        cs = B@np.array(vs).T
        Ms = Binter@cs
        mask = Ms[0] > 12
        cs = cs[:,mask]
        Ms = Ms[:,mask]
        
        if not any(mask):
            continue
    
        # compute the Ms, Ks
        Ks = Z@Ms
        assert np.allclose(Ks[1:]/gcd,Ks[1:]//gcd)
        Ks = Ks//gcd
    
        # check that K0 can be set...
        Qperp = -np.sum(Ks[1:]*Ms[1:], axis=0)
        divisible = ((Qperp-Qmax) % Ms[0]) == 0
        Qperp = Qperp[divisible]
        Ms = Ms[:,divisible]
        Ks = Ks[:,divisible]
    
        if verbosity >= 1:
            print(f"{Ms.shape[1]} passed tadpole cut...")
    
        # check that N has nonzero det N, check it has nonzero det
        for Q,K,M in zip(Qperp,Ks.T,Ms.T):
            N = (data.kappa_cob@M)[1:,1:]
            if np.linalg.matrix_rank(N)==N.shape[0]:
                pass
            else:
                continue
    
            # set K0
            K[0] = (Q-Qmax)//M[0]
    
            #print(f"K = {K.tolist()}")
            #print(f"M = {M.tolist()}")
            #print(f"p = {p.tolist()}")
            #print()
            pfv = diagnostics.PFV(data, K=K, M=M, silent=False)
            if not pfv.check_all():
                #print(pfv.Kprime)
                pass
            else:
                print(":)")
                good_pfvs.append(pfv)
t1 = time.time()

In [ ]:
print(t1-t0)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(pred,obs)
plt.xlabel('pred')
plt.ylabel('obs')
#plt.xlim([10**(-3), None])
#plt.xscale('log')
#plt.yscale('log')

In [ ]:
plt.hist(pred, bins=np.arange(min(pred)-0.5,max(pred)+1.5,2), histtype='step')
plt.hist(obs, bins=np.arange(min(obs)-0.5,max(obs)+1.5, 2),  histtype='step')
#plt.xscale('log')
plt.yscale('log')